In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from astropy.io import fits
import os
import numpy as np
import jax.numpy as jnp

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

import matplotlib.pyplot as plt

In [ ]:
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.mass.nfw import NFW_ELLIPSE, NFW_ELLIPSE_EINSTEIN
from gigalens.jax.profiles.mass.nfw_ellipse_slope import NFW_ELLIPSE_SLOPE
from gigalens.jax.profiles.mass.piemd import DPIE

from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.profiles.light.shapelets import Shapelets

from gigalens.jax.cosmo import wCDM_Cosmo

from gigalens.jax.scene_prob_model import Dataset, ImageData, ProbModel
from gigalens.simulator import SimulatorConfig

In [ ]:
from gigalens.jax.utils.grouped_priors import DiskEllipticity
from gigalens.jax.experimental.adaptive_supersample import AdaptiveImageData, plot_factor_map

In [ ]:
from translate_old_params import build
model, spec = build("MAP_best_31JulNFW_fixedcosmo_fixedLowZ.json")

In [ ]:
model.planes

In [ ]:
z1_2=0.962
z3=1.166
z4_5=1.432
z9=1.506
z7=1.627
z6=1.656
z12_13=3.086
z8=3.549
z11=4.090

In [ ]:
p = model.to_params({})
srclight = p['planes'][9]['light'][0]
srclight['center_x'] = 2.8828425
srclight['center_y'] = 1.1054275
srclight['n_sersic'] = 8.
srclight['Ie'] = 10.
p['planes'][7]['light'][2]['Ie']=100.

In [ ]:
from gigalens_research.plotting import plot_scene
from gigalens.jax.scene_simulator import SceneSimulator
cfg_k = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=None, likelihood_precision="float64")
sims = [SceneSimulator(model, cfg_k, sees=model.planes[i].light) for i in range(1, len(model.planes))]
figs = plot_scene(model, sims, p)#, fov_arcsec={2: 5.0, 7: 10.0}, center={1: (8.0, 3.5)})
plt.show()

In [ ]:
import os
from astropy.io import fits
def dataset_from_dir(path, ext):

    img_path = os.path.join(path, f"source{ext}.fits")
    with fits.open(img_path) as hdul:
        observed_image = jnp.array(hdul['DATA'].data.astype("float64"))

        error_map = jnp.array(np.sqrt(hdul['STAT'].data.astype("float64")))
        # background_rms = hdul['DATA'].header['BKG_RMS']
        # exp_time = hdul['PRIMARY'].header['EXPTIME']
            # if centroids is None: (self.centroids_x, self.centroids_y) = Table(hdul['CENTROIDS'].data)['centroid'].data.T
            # if centroids_error is None: self.centroids_error = Table(hdul['CENTROIDS'].data)['sky_covariance'].data
        psf = hdul['PSF'].data.astype(np.float64)
        mask = hdul['MASK'].data.astype(jnp.bool)
        # hot_pix = jnp.load(os.path.join(path, f"hot_pix.npy"))

    # mask = jnp.logical_and(mask, hot_pix)

    return observed_image, error_map, psf, mask

def ds(ext, sees):
    observed_image, error_map, psf, mask = dataset_from_dir(path, ext)

    cfg = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=psf, likelihood_precision="float64", conv_precision="float32")
    dset = ImageData(observed_image, cfg, error_map=error_map, mask=mask, sees=sees)
    return dset

path = "real_cutouts/"

from translate_old_params import parse, cutout_extensions
import json
spec = parse(json.load(open("MAP_best_31JulNFW_fixedcosmo_fixedLowZ.json")))

exts = cutout_extensions(spec)
source_planes = [p for p in model.planes if p.has_light]
assert len(exts) == len(source_planes), (
    f"{len(exts)} cutouts vs {len(source_planes)} source planes — "
    "zip would silently drop the tail")

base_datasets = [ds(ext, plane.light) for ext, plane in zip(exts, source_planes)]
base_prob_model = ProbModel(model, base_datasets, mode="forward")



In [ ]:
base_obs = [s.simulate(p) for s in base_prob_model.simulators]
adaptive_datasets = []
for ds, obs in zip(base_prob_model.datasets, base_obs):
    adaptive_datasets.append(AdaptiveImageData(obs, ds.sim_config, error_map=ds.error_map, mask=ds.mask, sees=ds.sees))
adaptive_prob_model = ProbModel(model, adaptive_datasets, mode="forward")

In [ ]:
9920 seconds
BKG_RMS

In [ ]:
from gigalens_research.plotting import plot_scene
sims = [s for s in prob_model.simulators]
figs = plot_scene(model, sims, p)#, fov_arcsec={2: 5.0, 7: 10.0}, center={1: (8.0, 3.5)})
plt.show()

In [ ]:
from gigalens_research.plotting import plot_scene
sims = [s for s in adaptive_prob_model.simulators]
figs = plot_scene(model, sims, p)#, fov_arcsec={2: 5.0, 7: 10.0}, center={1: (8.0, 3.5)})
plt.show()

In [ ]:
from gigalens.jax.analysis import diagnose_undersampling

In [ ]:
rep = diagnose_undersampling(adaptive_prob_model, p)

In [ ]:
for r in rep:
    r.plot()

In [ ]:
for d in adaptive_prob_model.datasets:
    plot_factor_map(d.adaptive_grid)

In [ ]:
p

In [ ]:
import importlib

import translate_old_params
importlib.reload(translate_old_params)
from translate_old_params import parse, cutout_extensions
spec = parse(json.load(open("MAP_best_31JulNFW_fixedcosmo_fixedLowZ.json")))
cutout_extensions(spec)